# MLOps Hands-on: Experiment Tracking with MLflow

**Dataset:** Kidney Disease Classification  
**Main topics:** Experiment Tracking, Runs, Parameters, Metrics, Artifacts, Model Logging and Run Comparison

---

## 🎯 Learning Objectives

By completing this notebook, students will be able to:

1. Understand why experiment tracking is required in MLOps.
2. Create an MLflow experiment.
3. Create and manage MLflow runs.
4. Log **parameters** such as model settings and hyperparameters.
5. Log **metrics** such as accuracy, precision, recall and F1-score.
6. Create and log **artifacts** such as confusion matrices and reports.
7. Compare multiple ML experiments in the MLflow UI.
8. Log a trained scikit-learn model to MLflow.
9. Understand the difference between an **experiment**, a **run**, a **parameter**, a **metric**, and an **artifact**.

> **Important:** This notebook is designed for learning MLOps concepts. The reported model performance is for demonstration only and should not be interpreted as clinical or real-world medical performance.

## 1. Prerequisites

Before running this notebook:

### Install required packages

Run in Anaconda Prompt / Terminal:

```bash
conda activate feast_lab
python -m pip install mlflow scikit-learn pandas matplotlib seaborn joblib
```

Verify MLflow:

```bash
mlflow --version
```

### Start the MLflow UI

Open a **separate terminal** in the same project folder and run:

```bash
mlflow ui
```

Then open the URL displayed by MLflow, usually:

```text
http://127.0.0.1:5000
```

### Recommended project structure

```text
MLOps_MLflow_Lab/
│
├── My_MLflow.ipynb
├── kidney_disease_cleaned.csv
└── mlruns/                 # created automatically by MLflow
```

> **Important:** Do not commit the `mlruns/` folder or large datasets to GitHub unless you intentionally want to version them. A `.gitignore` file is provided at the end of this notebook.

## 2. MLflow Concepts — The Big Picture

MLflow helps us keep track of ML experiments.

```text
                    MLflow Experiment
                           │
          ┌────────────────┼────────────────┐
          ↓                ↓                ↓
        Run 1            Run 2            Run 3
          │                │                │
     Parameters        Parameters       Parameters
     Metrics           Metrics          Metrics
     Artifacts         Artifacts        Artifacts
```

### Remember

| Concept | Simple meaning |
|---|---|
| **Experiment** | A collection of related ML runs |
| **Run** | One execution of an ML experiment |
| **Parameter** | A value/configuration we choose |
| **Metric** | A value measured after the model runs |
| **Artifact** | A file produced by the experiment |

### The key questions

- **Parameter:** What did I configure?
- **Metric:** How well did it perform?
- **Artifact:** What file did I produce?
- **Run:** What happened in this particular execution?

## 3. Import Libraries

We use:

- **pandas / NumPy** for data handling
- **scikit-learn** for preprocessing and models
- **matplotlib / seaborn** for visualizations
- **MLflow** for experiment tracking

The preprocessing is included inside the scikit-learn pipeline so that students can focus on the MLflow workflow.

In [ ]:
# ============================================
# CELL 1: IMPORT LIBRARIES
# ============================================

import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")
print("MLflow version:", mlflow.__version__)

## 4. Load the Dataset

We use the cleaned kidney disease dataset.

> **File requirement:** Keep `kidney_disease_cleaned.csv` in the same folder as this notebook.

The target column is:

```text
classification
```

The `id` column is removed later because it is an identifier rather than a useful predictive feature.

In [ ]:
# ============================================
# CELL 2: LOAD DATASET
# ============================================

DATA_FILE = "kidney_disease_cleaned.csv"

df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

## 5. Basic Dataset Check

Before training a model, inspect:

- dataset size
- column names
- missing values
- data types

This is not the main focus of the MLflow lab; it simply prepares the dataset for model training.

In [ ]:
# ============================================
# CELL 3: BASIC DATASET INFORMATION
# ============================================

print("Dataset Shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nData Types:")
print(df.dtypes)

## 6. Separate Features and Target

The target is `classification`.

```text
X → input features
y → target/output
```

We will train models using `X_train` and `y_train`, and evaluate them using `X_test` and `y_test`.

In [ ]:
# ============================================
# CELL 4: SEPARATE FEATURES AND TARGET
# ============================================

X = df.drop("classification", axis=1)
y = df["classification"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget classes:")
print(y.value_counts())

## 7. Remove the ID Column

The `id` column identifies a record. It should not normally be treated as a predictive feature.

```text
id → identifier
age, bp, sg, ... → features
classification → target
```

In [ ]:
# ============================================
# CELL 5: REMOVE ID COLUMN
# ============================================

if "id" in X.columns:
    X = X.drop("id", axis=1)
    print("ID column removed.")
else:
    print("ID column not found.")

print("\nRemaining feature columns:")
print(X.columns.tolist())

## 8. Identify Numerical and Categorical Features

Different types of data require different preprocessing:

- **Numerical:** imputation + scaling
- **Categorical:** imputation + one-hot encoding

This preprocessing will be included in every model pipeline.

In [ ]:
# ============================================
# CELL 6: IDENTIFY COLUMN TYPES
# ============================================

numeric_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numerical columns:")
print(numeric_columns)

print("\nCategorical columns:")
print(categorical_columns)

## 9. Create the Preprocessing Pipeline

### Numerical pipeline

```text
Missing values → Mean imputation → Standardization
```

### Categorical pipeline

```text
Missing values → Most-frequent imputation → One-hot encoding
```

Using a pipeline prevents us from manually repeating preprocessing for every model.

In [ ]:
# ============================================
# CELL 7: PREPROCESSING PIPELINE
# ============================================

# Numerical features:
# 1. Fill missing values using the mean.
# 2. Standardize the numerical values.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

# Categorical features:
# 1. Fill missing values using the most frequent value.
# 2. Convert categories into numerical one-hot encoded columns.
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both preprocessing pipelines.
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])

print("Preprocessing pipeline created.")

## 10. Train-Test Split

We use:

- **80%** for training
- **20%** for testing
- `random_state=42` for reproducibility
- `stratify=y` to preserve the class distribution

In [ ]:
# ============================================
# CELL 8: TRAIN-TEST SPLIT
# ============================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# PART A — BASIC MLFLOW

## 11. Configure the MLflow Experiment

An **experiment** groups related runs.

We will create:

```text
Kidney Disease Classification
```

Inside this experiment we will create multiple runs:

```text
Run 1 → Logistic Regression
Run 2 → Logistic Regression with a different C
Run 3 → Random Forest
Run 4 → Decision Tree
Run 5 → SVM
```

This allows us to compare experiments systematically.

In [ ]:
# ============================================
# CELL 9: CONFIGURE MLFLOW
# ============================================

EXPERIMENT_NAME = "Kidney Disease Classification"

# Use a local tracking directory so the experiment
# can be viewed by the local MLflow UI.
mlflow.set_tracking_uri("file:./mlruns")

# Create the experiment if it does not exist,
# or select it if it already exists.
mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow experiment selected:")
print(EXPERIMENT_NAME)

## 12. Create the First Model — Logistic Regression

We start with a simple baseline model.

The model pipeline contains:

```text
Preprocessing
     ↓
Logistic Regression
```

The parameter `C` controls the regularization strength used by Logistic Regression.

We will record `C` in MLflow so we can later compare different configurations.

In [ ]:
# ============================================
# CELL 10: LOGISTIC REGRESSION MODEL
# ============================================

model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(
        C=1.0,
        max_iter=1000
    ))
])

print(model)

## 13. First MLflow Run — Parameters + Metrics

This is the most important MLflow cell.

### What are we logging?

**Parameters**

```text
model
C
test_size
random_state
```

**Metrics**

```text
accuracy
precision
recall
f1_score
```

### Important

`mlflow.start_run()` creates one MLflow **Run**.

Everything logged inside the `with` block belongs to that run.

In [ ]:
# ============================================
# CELL 11: FIRST MLFLOW RUN
# ============================================

with mlflow.start_run(
    run_name="Logistic Regression - Run 1"
):

    # ----------------------------------------
    # PARAMETERS
    # What did we configure?
    # ----------------------------------------

    mlflow.log_params({
        "model": "Logistic Regression",
        "C": 1.0,
        "test_size": 0.20,
        "random_state": 42
    })

    # ----------------------------------------
    # TRAIN
    # ----------------------------------------

    model.fit(X_train, y_train)

    # ----------------------------------------
    # PREDICT
    # ----------------------------------------

    predictions = model.predict(X_test)

    # ----------------------------------------
    # CALCULATE METRICS
    # What did we measure?
    # ----------------------------------------

    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(
        y_test,
        predictions,
        average="weighted"
    )

    recall = recall_score(
        y_test,
        predictions,
        average="weighted"
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="weighted"
    )

    # ----------------------------------------
    # LOG METRICS
    # ----------------------------------------

    mlflow.log_metrics({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    })

    print("Model: Logistic Regression")
    print("Accuracy :", accuracy)
    print("Precision:", precision)
    print("Recall   :", recall)
    print("F1 Score :", f1)

### 🔎 Check the MLflow UI

Refresh the MLflow browser.

Go to:

```text
Kidney Disease Classification
        ↓
Runs
        ↓
Logistic Regression - Run 1
```

Look at:

- **Parameters**
- **Metrics**

### Student question

> What is the difference between a parameter and a metric?

**Answer:**

- Parameter → value/configuration chosen for the experiment.
- Metric → performance value measured after execution.

# PART B — COMPLETE TRACKING

## 14. Second Run — Change a Parameter

Now we change only:

```text
C = 0.1
```

This creates a new run.

The purpose is to demonstrate that MLflow keeps the two experiments separately, making comparison possible.

In [ ]:
# ============================================
# CELL 12: LOGISTIC REGRESSION - RUN 2
# ============================================

model_2 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(
        C=0.1,
        max_iter=1000
    ))
])

with mlflow.start_run(
    run_name="Logistic Regression - Run 2"
):

    # Log the configuration used in this run.
    mlflow.log_params({
        "model": "Logistic Regression",
        "C": 0.1,
        "test_size": 0.20,
        "random_state": 42
    })

    # Train the model.
    model_2.fit(X_train, y_train)

    # Generate predictions.
    predictions_2 = model_2.predict(X_test)

    # Calculate evaluation metrics.
    accuracy_2 = accuracy_score(y_test, predictions_2)

    precision_2 = precision_score(
        y_test,
        predictions_2,
        average="weighted"
    )

    recall_2 = recall_score(
        y_test,
        predictions_2,
        average="weighted"
    )

    f1_2 = f1_score(
        y_test,
        predictions_2,
        average="weighted"
    )

    # Log all metrics to this run.
    mlflow.log_metrics({
        "accuracy": accuracy_2,
        "precision": precision_2,
        "recall": recall_2,
        "f1_score": f1_2
    })

    print("Accuracy :", accuracy_2)
    print("Precision:", precision_2)
    print("Recall   :", recall_2)
    print("F1 Score :", f1_2)

## 15. Compare the Two Logistic Regression Runs

In the MLflow UI, compare:

```text
Run 1 → C = 1.0
Run 2 → C = 0.1
```

Ask students:

> **Did changing the parameter change the performance?**

This is the basic idea behind experiment tracking and later leads naturally to **hyperparameter tuning**.

## 16. Create an Artifact — Confusion Matrix

An **artifact** is a file produced by an ML experiment.

Examples:

- confusion matrix image
- classification report
- CSV file
- trained model
- feature importance plot

Here we create a confusion matrix image.

In [ ]:
# ============================================
# CELL 13: CREATE CONFUSION MATRIX
# ============================================

# Create a confusion matrix using the predictions
# from Logistic Regression Run 2.
cm = confusion_matrix(
    y_test,
    predictions_2
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()

# Save the figure as a file.
# This file will become an MLflow artifact.
plt.savefig(
    "confusion_matrix.png",
    dpi=300
)

plt.show()

### What happened?

We now have a physical file:

```text
confusion_matrix.png
```

But simply creating the file does **not** put it into MLflow.

We must explicitly log it using:

```python
mlflow.log_artifact("confusion_matrix.png")
```

In [ ]:
# ============================================
# CELL 14: CREATE CLASSIFICATION REPORT
# ============================================

# Generate a text report containing
# precision, recall, F1-score and support.
report = classification_report(
    y_test,
    predictions_2
)

print(report)

# Save the report as a text file.
with open(
    "classification_report.txt",
    "w"
) as file:
    file.write(report)

print("Classification report saved.")

## 17. Complete Tracking Run — Parameters + Metrics + Artifacts

This run demonstrates the complete tracking concept.

```text
                 MLflow Run
                     │
       ┌─────────────┼─────────────┐
       ↓             ↓             ↓
 Parameters       Metrics       Artifacts
       ↓             ↓             ↓
      C            Accuracy       PNG
    test_size      Precision      TXT
   random_state    Recall
                   F1
```

In [ ]:
# ============================================
# CELL 15: COMPLETE TRACKING RUN
# ============================================

with mlflow.start_run(
    run_name="Logistic Regression - Complete Tracking"
):

    # -------------------------------
    # PARAMETERS
    # -------------------------------

    mlflow.log_params({
        "model": "Logistic Regression",
        "C": 0.1,
        "test_size": 0.20,
        "random_state": 42
    })

    # -------------------------------
    # METRICS
    # -------------------------------

    mlflow.log_metrics({
        "accuracy": accuracy_2,
        "precision": precision_2,
        "recall": recall_2,
        "f1_score": f1_2
    })

    # -------------------------------
    # ARTIFACTS
    # -------------------------------

    mlflow.log_artifact(
        "confusion_matrix.png"
    )

    mlflow.log_artifact(
        "classification_report.txt"
    )

    print("Complete MLflow run logged successfully.")

### 🔎 Check the MLflow UI again

Open:

```text
Logistic Regression - Complete Tracking
```

Now the run should contain:

```text
Parameters
├── model
├── C
├── test_size
└── random_state

Metrics
├── accuracy
├── precision
├── recall
└── f1_score

Artifacts
├── confusion_matrix.png
└── classification_report.txt
```

> **Key lesson:** `mlflow.log_param()`, `mlflow.log_metric()`, and `mlflow.log_artifact()` control what is stored with the run.

# PART C — COMPARE EXPERIMENTS

## 18. Train Multiple Models

Now we will track different algorithms as separate MLflow runs:

1. Logistic Regression
2. Random Forest
3. Decision Tree
4. SVM

The objective is not to find the clinically best model. The objective is to learn how MLflow helps us **record and compare different experiments**.

In [ ]:
# ============================================
# CELL 16: RANDOM FOREST
# ============================================

rf_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        random_state=42
    ))
])

with mlflow.start_run(
    run_name="Random Forest"
):

    # Parameters
    mlflow.log_params({
        "model": "Random Forest",
        "n_estimators": 100,
        "max_depth": 10,
        "random_state": 42,
        "test_size": 0.20
    })

    # Train
    rf_model.fit(X_train, y_train)

    # Predict
    rf_predictions = rf_model.predict(X_test)

    # Metrics
    rf_accuracy = accuracy_score(y_test, rf_predictions)

    rf_precision = precision_score(
        y_test,
        rf_predictions,
        average="weighted"
    )

    rf_recall = recall_score(
        y_test,
        rf_predictions,
        average="weighted"
    )

    rf_f1 = f1_score(
        y_test,
        rf_predictions,
        average="weighted"
    )

    # Log metrics
    mlflow.log_metrics({
        "accuracy": rf_accuracy,
        "precision": rf_precision,
        "recall": rf_recall,
        "f1_score": rf_f1
    })

    print("Random Forest")
    print("Accuracy :", rf_accuracy)
    print("Precision:", rf_precision)
    print("Recall   :", rf_recall)
    print("F1 Score :", rf_f1)

In [ ]:
# ============================================
# CELL 17: DECISION TREE
# ============================================

dt_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ))
])

with mlflow.start_run(
    run_name="Decision Tree"
):

    # Parameters
    mlflow.log_params({
        "model": "Decision Tree",
        "max_depth": 5,
        "random_state": 42,
        "test_size": 0.20
    })

    # Train
    dt_model.fit(X_train, y_train)

    # Predict
    dt_predictions = dt_model.predict(X_test)

    # Metrics
    dt_accuracy = accuracy_score(y_test, dt_predictions)

    dt_precision = precision_score(
        y_test,
        dt_predictions,
        average="weighted"
    )

    dt_recall = recall_score(
        y_test,
        dt_predictions,
        average="weighted"
    )

    dt_f1 = f1_score(
        y_test,
        dt_predictions,
        average="weighted"
    )

    # Log metrics
    mlflow.log_metrics({
        "accuracy": dt_accuracy,
        "precision": dt_precision,
        "recall": dt_recall,
        "f1_score": dt_f1
    })

    print("Decision Tree")
    print("Accuracy :", dt_accuracy)
    print("Precision:", dt_precision)
    print("Recall   :", dt_recall)
    print("F1 Score :", dt_f1)

In [ ]:
# ============================================
# CELL 18: SUPPORT VECTOR MACHINE (SVM)
# ============================================

svm_model = Pipeline([
    ("preprocessing", preprocessor),
    ("model", SVC(
        C=1.0,
        kernel="rbf"
    ))
])

with mlflow.start_run(
    run_name="SVM"
):

    # Parameters
    mlflow.log_params({
        "model": "SVM",
        "C": 1.0,
        "kernel": "rbf",
        "test_size": 0.20
    })

    # Train
    svm_model.fit(X_train, y_train)

    # Predict
    svm_predictions = svm_model.predict(X_test)

    # Metrics
    svm_accuracy = accuracy_score(y_test, svm_predictions)

    svm_precision = precision_score(
        y_test,
        svm_predictions,
        average="weighted"
    )

    svm_recall = recall_score(
        y_test,
        svm_predictions,
        average="weighted"
    )

    svm_f1 = f1_score(
        y_test,
        svm_predictions,
        average="weighted"
    )

    # Log metrics
    mlflow.log_metrics({
        "accuracy": svm_accuracy,
        "precision": svm_precision,
        "recall": svm_recall,
        "f1_score": svm_f1
    })

    print("SVM")
    print("Accuracy :", svm_accuracy)
    print("Precision:", svm_precision)
    print("Recall   :", svm_recall)
    print("F1 Score :", svm_f1)

## 19. Create a Model Comparison Table

MLflow is the main experiment tracking tool. We also create a simple pandas table so students can understand the same comparison programmatically.

Later, the same information can be compared directly inside the MLflow UI.

In [ ]:
# ============================================
# CELL 19: COMPARE MODELS
# ============================================

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Decision Tree",
        "SVM"
    ],
    "Accuracy": [
        accuracy_2,
        rf_accuracy,
        dt_accuracy,
        svm_accuracy
    ],
    "Precision": [
        precision_2,
        rf_precision,
        dt_precision,
        svm_precision
    ],
    "Recall": [
        recall_2,
        rf_recall,
        dt_recall,
        svm_recall
    ],
    "F1 Score": [
        f1_2,
        rf_f1,
        dt_f1,
        svm_f1
    ]
})

comparison = comparison.sort_values(
    "F1 Score",
    ascending=False
).reset_index(drop=True)

comparison

## 20. Save the Comparison Table as an Artifact

A CSV file generated by an experiment is also an **artifact**.

We save:

```text
model_comparison.csv
```

and then log it to MLflow.

In [ ]:
# ============================================
# CELL 20: SAVE MODEL COMPARISON
# ============================================

comparison.to_csv(
    "model_comparison.csv",
    index=False
)

print("Model comparison saved as model_comparison.csv")

In [ ]:
# ============================================
# CELL 21: LOG COMPARISON ARTIFACT
# ============================================

best_row = comparison.iloc[0]

with mlflow.start_run(
    run_name="Model Comparison Summary"
):

    # Dataset and experiment information
    mlflow.log_params({
        "dataset": DATA_FILE,
        "models_compared": 4
    })

    # Log the best observed values from this classroom experiment.
    mlflow.log_metrics({
        "best_accuracy": float(best_row["Accuracy"]),
        "best_f1_score": float(best_row["F1 Score"])
    })

    # Log the complete comparison table.
    mlflow.log_artifact(
        "model_comparison.csv"
    )

    print("Best model:", best_row["Model"])
    print("Comparison artifact logged.")

## 21. Log the Best Model

Now we demonstrate MLflow model logging.

This is different from:

```python
mlflow.log_artifact("model.pkl")
```

`mlflow.sklearn.log_model()` logs the trained scikit-learn model using the MLflow scikit-learn model flavor.

### Classroom note

We use `cloudpickle` here because the installed MLflow version may use `skops` by default and can request trusted types for some pipelines. `cloudpickle` is convenient for this classroom demonstration, but serialized Python objects should only be loaded from trusted sources.

> **Model Registry is NOT covered yet.** It is a later topic under Model Management.

In [ ]:
# ============================================
# CELL 22: SELECT AND LOG THE BEST MODEL
# ============================================

best_model_name = best_row["Model"]

# Select the already-trained model object.
if best_model_name == "Logistic Regression":
    best_model = model_2

elif best_model_name == "Random Forest":
    best_model = rf_model

elif best_model_name == "Decision Tree":
    best_model = dt_model

else:
    best_model = svm_model


with mlflow.start_run(
    run_name="Best Model - Final"
):

    # Identify the selected model.
    mlflow.log_param(
        "selected_model",
        best_model_name
    )

    # Log its evaluation results.
    mlflow.log_metrics({
        "accuracy": float(best_row["Accuracy"]),
        "precision": float(best_row["Precision"]),
        "recall": float(best_row["Recall"]),
        "f1_score": float(best_row["F1 Score"])
    })

    # Log the model using the MLflow scikit-learn flavor.
    # cloudpickle is used here for compatibility with this
    # classroom environment.
    mlflow.sklearn.log_model(
        sk_model=best_model,
        name="kidney_disease_model",
        serialization_format="cloudpickle"
    )

    print("Best model logged successfully!")
    print("Model:", best_model_name)

## 22. Final Result

Let's display the final model comparison.

**Important:** The highest score in this notebook is only the result of this particular train/test split and configuration. Students should not treat it as a general claim about model quality.

In [ ]:
# ============================================
# CELL 23: FINAL MODEL COMPARISON
# ============================================

print("=" * 60)
print("FINAL MODEL COMPARISON")
print("=" * 60)

display(comparison)

print("\nBest Model:")
print(best_row["Model"])

print("\nBest Accuracy:")
print(best_row["Accuracy"])

print("\nBest F1 Score:")
print(best_row["F1 Score"])

# 23. Explore the MLflow UI

Refresh the MLflow browser and open:

```text
Kidney Disease Classification
```

## Check the Runs

You should see multiple runs such as:

```text
Logistic Regression - Run 1
Logistic Regression - Run 2
Logistic Regression - Complete Tracking
Random Forest
Decision Tree
SVM
Model Comparison Summary
Best Model - Final
```

## For each run, inspect:

### Parameters
What configuration was used?

### Metrics
How well did the model perform?

### Artifacts
What files were generated?

### Model
Which trained model was logged?

---

## 🎓 Questions for Students

1. What is the difference between an **experiment** and a **run**?
2. Why is `C` a parameter rather than a metric?
3. Why is accuracy a metric?
4. What is an artifact?
5. Why do we create separate runs for different models?
6. How can MLflow help reproduce an experiment?
7. What happens if we change `C` from `1.0` to `0.1`?
8. Why should we not blindly choose a model only because one test split produced the highest accuracy?

# 24. Mini Assignment

## Task 1 — Add Another Model

Add one more classifier, for example:

```text
K-Nearest Neighbors
```

Log:

- model name
- important hyperparameters
- accuracy
- precision
- recall
- F1-score

---

## Task 2 — Add an Artifact

Create and log:

```text
confusion_matrix.png
```

for your new model.

---

## Task 3 — Compare Runs

Use the MLflow UI to answer:

> Which model performed best according to F1-score?

---

## Task 4 — Change a Hyperparameter

For Random Forest, try:

```text
n_estimators = 50
n_estimators = 100
n_estimators = 200
```

Create a separate MLflow run for each configuration.

### Goal

Observe how changing a parameter creates different experiment results.

---

# Key Learning

```text
                MLFLOW RUN
                    │
       ┌────────────┼────────────┐
       ↓            ↓            ↓
 PARAMETERS       METRICS     ARTIFACTS
       ↓            ↓            ↓
 What did I     How well did   What files
 configure?     it perform?    did I create?
       │            │            │
       └────────────┼────────────┘
                    ↓
             EXPERIMENT TRACKING
```

# 25. GitHub Preparation

A simple GitHub repository can contain:

```text
MLOps-MLflow-Lab/
│
├── My_MLflow.ipynb
├── kidney_disease_cleaned.csv       # include only if permitted
├── README.md
└── .gitignore
```

## Recommended `.gitignore`

Create a file named `.gitignore`:

```text
# MLflow local tracking data
mlruns/

# Python
__pycache__/
*.py[cod]

# Jupyter checkpoints
.ipynb_checkpoints/

# Generated MLflow artifacts / local outputs
confusion_matrix.png
classification_report.txt
model_comparison.csv

# Trained model files
*.pkl

# Virtual environments
.venv/
venv/
env/
```

> If the dataset has licensing, privacy, or redistribution restrictions, do **not** upload it to a public GitHub repository. In that case, include instructions in the README explaining where students can obtain the dataset.

# 26. Suggested README Summary

You can use the following description in GitHub:

> **MLOps Hands-on: Experiment Tracking with MLflow**
>
> This notebook demonstrates experiment tracking using MLflow with a kidney disease classification dataset. Students learn how to create experiments and runs, log parameters and metrics, create and log artifacts, compare multiple machine-learning models, and log a trained scikit-learn model.
>
> **Topics covered**
> - MLflow Experiment
> - MLflow Run
> - Parameter Logging
> - Metric Logging
> - Artifact Logging
> - Model Comparison
> - Scikit-learn Model Logging
>
> **Models**
> - Logistic Regression
> - Random Forest
> - Decision Tree
> - Support Vector Machine (SVM)
>
> **Environment**
> - Python
> - pandas
> - scikit-learn
> - MLflow
> - matplotlib
> - seaborn

# End of Lab

## What students should remember

**Experiment** → collection of related runs  
**Run** → one execution  
**Parameter** → configuration/input  
**Metric** → measured result  
**Artifact** → generated file  
**MLflow** → keeps these pieces together so experiments can be tracked and compared

### Next MLOps topic

➡️ **Hyperparameter Tuning Basics**

The natural connection is:

```text
Manual Experiments
      ↓
MLflow Tracking
      ↓
Many Hyperparameter Configurations
      ↓
Automated Hyperparameter Tuning
      ↓
Compare Runs
      ↓
Select Best Model
```